# USB CT: comparing 3D reconstructions from sparse views

**How do FDK, SIRT and FaCT-GS differ when fewer projection views are available?** This example uses a USB cone-beam CT scan to compare reconstructions from 25, 50 and 75 training views. A shared set of 20 held-out views evaluates projection consistency.

We use CIL to read and preprocess the data and establish FDK and SIRT baselines, then run FaCT-GS with the same splits and geometry scale. Reconstruction slices follow each method, followed by projection metrics and historical experiment figures.

**Historical CIL version:** `26.0.1.dev11+gb39f95799`; the installation below uses `26.0.0`. **Data:** `gruppe 4_tomo-A.txrm`, available through CIL's USB example download (Section 2), not bundled here. **Authorship, contribution licence and data reuse terms:** to be confirmed for this local draft.

Current results and historical results are labelled separately. The configuration below enables all three reconstruction methods; disable any methods you do not want to run. The CIL workflow uses in-memory data without intermediate files. The upstream FaCT-GS trainer saves models, but projections and results are passed in memory within this notebook.


## 0. Set up the environment

This example uses **Linux and an NVIDIA GPU**, with TIGRE as the CIL projection backend. The reference setup is Python 3.10, PyTorch 2.4.1 and CUDA 11.8. The full FaCT-GS workflow also requires the CUDA Toolkit (including `nvcc`) and a compatible C/C++ compiler.

Run these commands in a terminal in the notebook directory:

```bash
conda env create -f environment.yml
conda activate usb-showcase
python -m pip install torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu118
python -m pip install -r requirements.txt --index-url https://pypi.org/simple
bash install_extensions.sh
```

Select the **Python kernel for `usb-showcase`** before running Section 1. To run only FDK/SIRT in an existing CIL/TIGRE environment, leave FaCT-GS disabled.

Conda only creates the Python environment; pip installs the remaining dependencies listed in `requirements.txt`. CIL is built from its official v26.0.0 source. Keep `third_party/fact-gs`, which contains the extension sources. If the compiler cannot find CUDA, set `CUDA_HOME` to your CUDA Toolkit directory and add its `bin` directory to `PATH`.

Set the data directory in Section 1. Section 2 downloads and reads the USB example through CIL and preprocesses it.


## 1. Configure the data and methods

Set the data directory, number of training views and GPU. Start with 25 views, enable FDK, SIRT and metrics as needed, and run the cells in order. After changing `VIEWS`, rerun from Section 3. The CIL workflow neither writes intermediate files nor imports this project's helper scripts.

Section 7 adapts the in-memory data and calls the upstream FaCT-GS trainer. It is disabled by default.

Use Section 10 for the complete 25/50/75-view comparison. First run Sections 1–8 to define the functions (the individual reconstruction switches may remain disabled), then enable `RUN_ALL_VIEWS`.


In [ ]:
from pathlib import Path
import os, time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

CASE = Path.cwd().resolve()
DATA_DIR = CASE / 'dataset'  # Directory for downloaded CIL example data
VIEWS = 25
GPU = '1'  # Select an available GPU; restart the kernel after changing this.
RUN_FDK = True
RUN_SIRT = True
RUN_FACTGS = True
RUN_METRICS = True
os.environ['CUDA_VISIBLE_DEVICES'] = GPU
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

# Volumes from this run; rerunning this cell clears them.
reconstructed_volumes = {}


## 2. Preprocess the data and compare before and after

Use CIL's `dataexample.USB` to obtain the [USB example dataset](https://zenodo.org/records/4822516). The first run downloads it to `DATA_DIR`; subsequent runs reuse the local data. Read the complete scan, take the negative logarithm of the transmission once, and crop 200 pixels from each horizontal edge. `ZEISSDataReader` already applies reference-image normalisation and the recorded integer-pixel shifts, so no further normalisation is needed. Retain negative attenuation values and apply no additional denoising.

The downloaded scan is stored on disk; preprocessing runs in memory without batching or intermediate files. After cropping, average non-overlapping 4×4 blocks and increase the detector pixel spacing by a factor of four. Later sections reuse this in-memory data without further downsampling.


In [ ]:
from cil.utilities import dataexample
from cil.processors import TransmissionAbsorptionConverter, Slicer

# Download the USB example once; skip if its directory already exists.
dataexample.USB.download_data(str(DATA_DIR), prompt=False)

# Read all transmission projections and keep the first view for comparison.
transmission = dataexample.USB.get(str(DATA_DIR))
before = transmission.get_slice(angle=0).as_array().copy()

# Convert transmission to attenuation line integrals with one negative logarithm.
absorption = TransmissionAbsorptionConverter(
    white_level=1.0, min_intensity=1e-6, accelerated=False,
)(transmission)

# Crop 200 pixels from each horizontal edge, preserving CIL data and geometry.
preprocessed = Slicer(roi={'horizontal': (200, -200)})(absorption)
# Average 4x4 blocks and update the detector geometry accordingly.
from cil.framework import AcquisitionData
cropped = preprocessed.as_array()
count, height, width = cropped.shape
averaged = cropped.reshape(count, height // 4, 4, width // 4, 4).mean(axis=(2, 4))
geometry = preprocessed.geometry.copy()
geometry.set_panel([width // 4, height // 4],
                   pixel_size=[geometry.pixel_size_h * 4, geometry.pixel_size_v * 4],
                   origin=geometry.config.panel.origin)
preprocessed = AcquisitionData(averaged, geometry=geometry)
after = preprocessed.get_slice(angle=0).as_array()


### Before and after preprocessing: the same view

The left image shows transmission returned by the reader after reference-image normalisation, not raw detector counts. The right image shows attenuation line integrals after the negative logarithm, cropping and 4×4 averaging. These are different physical quantities, so they use separate display ranges and colour bars; their brightness cannot be compared directly. Orange dashed lines mark the crop boundaries. Display windows affect only the plots, not the data.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, image, title in zip(
    axes, [before, after],
    ['Before: transmission', 'After: negative log + crop + 4x4 mean'],
):
    low, high = np.percentile(image, [1, 99])
    im = ax.imshow(image, cmap='gray', origin='lower', vmin=low, vmax=high)
    ax.set_title(f'{title} ({image.shape[0]} x {image.shape[1]})')
    ax.set_xlabel('Horizontal pixel')
    ax.set_ylabel('Vertical pixel')
    fig.colorbar(im, ax=ax, shrink=0.8)

axes[0].axvline(199.5, color='orange', linestyle='--')
axes[0].axvline(before.shape[1] - 200.5, color='orange', linestyle='--')
plt.tight_layout()
plt.show()


## 3. Split the training and test views

Select the fixed set of 25, 50 or 75 training projections according to `VIEWS`. All three experiments share 20 test views, which are excluded from reconstruction and used for projection evaluation.

The indices below reproduce the historical experiment splits. They exclude the 360° endpoint, which nearly duplicates the starting view. The test set is disjoint from all three training sets.


In [ ]:
# Fixed original view indices, matching the historical experiments.
train_splits = {75: [0, 11, 21, 32, 43, 53, 64, 75, 85, 96, 107, 117, 128, 139, 149, 160, 171, 181, 192, 203, 213, 224, 235, 245, 256, 267, 277, 288, 299, 309, 320, 331, 341, 352, 363, 373, 384, 395, 405, 416, 427, 437, 448, 459, 469, 480, 491, 501, 512, 523, 533, 544, 555, 565, 576, 587, 597, 608, 619, 629, 640, 651, 661, 672, 683, 693, 704, 715, 725, 736, 747, 757, 768, 779, 789], 50: [0, 16, 32, 48, 64, 80, 96, 112, 128, 144, 160, 176, 192, 208, 224, 240, 256, 272, 288, 304, 320, 336, 352, 368, 384, 400, 416, 432, 448, 464, 480, 496, 512, 528, 544, 560, 576, 592, 608, 624, 640, 656, 672, 688, 704, 720, 736, 752, 768, 784], 25: [0, 32, 64, 96, 128, 160, 192, 224, 256, 288, 320, 352, 384, 416, 448, 480, 512, 544, 576, 608, 640, 672, 704, 736, 768]}
train_indices = np.asarray(train_splits[VIEWS])
test_indices = np.asarray([1, 41, 84, 126, 168, 210, 252, 294, 337, 379, 420, 462, 505, 547, 589, 631, 673, 714, 758, 799])
assert len(train_indices) == VIEWS and len(test_indices) == 20
assert not np.intersect1d(train_indices, test_indices).size
print('Training views:', len(train_indices), '| Test views:', len(test_indices))

# Subsequent reconstruction uses the in-memory data from Section 2.
projections = preprocessed.as_array()
angles = np.asarray(preprocessed.geometry.angles)


## 4. Prepare the reconstruction inputs

We answer two questions in order: **which projections are used, and which 3D grid is reconstructed?** First prepare the training data and voxel grid, then apply the coordinate scale used in the historical FaCT-GS comparison. Test-view geometry is created later, in Section 8.

### 4.1 Select training projections and angles

The reader already provides the acquisition geometry. Copy it and select the training angles. `data` binds the training projections to their acquisition geometry without additional preprocessing.


In [ ]:
def select_training_data(preprocessed, indices):
    from cil.framework import AcquisitionData
    geometry = preprocessed.geometry.copy()
    geometry.set_angles(preprocessed.geometry.angles[indices],
                        initial_angle=geometry.config.angles.initial_angle,
                        angle_unit=geometry.config.angles.angle_unit)
    return AcquisitionData(preprocessed.as_array()[indices].copy(), geometry=geometry)

data = select_training_data(preprocessed, train_indices)


### 4.2 Define the output voxel grid

Estimate the reconstruction region from the detector coverage and magnification. Use 128 voxels along the longest dimension and scale the other dimensions proportionally. At this stage, use the scan's original physical units. `ig` specifies the volume dimensions and voxel spacing.


In [ ]:
def make_volume_grid(ag):
    from cil.framework import ImageGeometry
    width, height = ag.pixel_num_h, ag.pixel_num_v
    sizes = np.array([width * ag.pixel_size_h, width * ag.pixel_size_h,
                      height * ag.pixel_size_v]) / ag.magnification
    counts = np.maximum(2, np.rint(sizes / max(sizes) * 128).astype(int))
    spacing = sizes / counts
    ig = ImageGeometry(voxel_num_x=int(counts[0]), voxel_num_y=int(counts[1]),
                       voxel_num_z=int(counts[2]), voxel_size_x=spacing[0],
                       voxel_size_y=spacing[1], voxel_size_z=spacing[2])
    return ig, sizes, counts

ig, sizes, counts = make_volume_grid(data.geometry)


### 4.3 Coordinate convention for the historical comparison

**This step is not required by CIL.** The preceding steps already define the reconstruction inputs. The historical FaCT-GS experiments set the longest dimension of the reconstruction region to 2. Scale the acquisition geometry and voxel spacing together to preserve that convention; projection values and voxel counts remain unchanged.

Keeping this normalisation preserves the numerical settings of the existing experiments. It is a consistent choice of units made before reconstruction, not a change to apply to only one geometry afterwards. Reconstructed attenuation coefficients are consequently expressed in inverse normalised length units.


In [ ]:
def normalize_geometry(data, sizes, counts):
    from cil.framework import AcquisitionData, ImageGeometry
    geometry_scale = 2 / max(sizes)
    ag = data.geometry.copy()
    for component in (ag.config.system.source, ag.config.system.detector,
                      ag.config.system.rotation_axis):
        component.position = component.position * geometry_scale
    ag.config.panel.pixel_size *= geometry_scale
    ag.config.units = 'normalized distance'
    spacing = sizes * geometry_scale / counts
    ig = ImageGeometry(voxel_num_x=int(counts[0]), voxel_num_y=int(counts[1]),
                       voxel_num_z=int(counts[2]), voxel_size_x=spacing[0],
                       voxel_size_y=spacing[1], voxel_size_z=spacing[2])
    data = AcquisitionData(data.as_array().copy(), geometry=ag)
    return data, ig

data, ig = normalize_geometry(data, sizes, counts)


## 5. Run FDK independently

For this 3D cone-beam geometry, CIL/TIGRE's `FBP` executes FDK. Timing below covers reconstruction only, excluding output saving and forward projection.

FDK provides a fast analytical baseline. Inspect the streaks and structural boundaries under sparse sampling.


In [ ]:
def reconstruct_fdk(data, ig):
    from cil.plugins.tigre import FBP
    assert data is not None, 'Run the geometry setup cells first.'
    start = time.perf_counter()
    fdk = FBP(ig, data.geometry)(data)
    elapsed = time.perf_counter() - start
    fdk_volume = np.asarray(fdk.as_array(), dtype=np.float32)
    return fdk_volume

if RUN_FDK:
    reconstructed_volumes['fdk'] = reconstruct_fdk(data, ig)


### Inspect the current FDK reconstruction

Display central slices along the three axes directly from the in-memory volume. The display window for an individual method may differ from the final shared window; use the shared-window comparison below to compare methods.


In [ ]:
if RUN_FDK:
    methods = ['fdk']
    volumes = {m: reconstructed_volumes[m] for m in methods if m in reconstructed_volumes}
    if volumes:
        low, high = np.percentile(np.concatenate([v.ravel()[::16] for v in volumes.values()]), [1, 99.5])
        fig, axes = plt.subplots(len(volumes), 3, figsize=(12, 3.5 * len(volumes)), squeeze=False)
        for row, (method, volume) in enumerate(volumes.items()):
            for axis, label in enumerate(['Z', 'Y', 'X']):
                view = np.take(volume, volume.shape[axis] // 2, axis=axis)
                axes[row, axis].imshow(view, cmap='gray', origin='lower', vmin=low, vmax=high)
                axes[row, axis].set_title(f'{method.upper()} | {label} centre')
                axes[row, axis].axis('off')
        plt.tight_layout()
        plt.show()


## 6. Run SIRT independently

Initialise a zero volume, impose non-negativity and run **1000 iterations**. Recompute `0.5 * sum((Ax-b)**2)` from the current solution every ten iterations, avoiding a mixture of the initial residual and CIL's cached weighted residual in the loss curve.

SIRT updates the volume by repeatedly comparing forward projections with measured projections. Compare this non-negative iterative reconstruction with FDK. A fixed iteration count does not imply convergence.


In [ ]:
def reconstruct_sirt(data, ig, iterations=1000):
    from cil.plugins.tigre import ProjectionOperator
    from cil.optimisation.algorithms import SIRT
    assert data is not None, 'Run the geometry setup cells first.'
    class ResidualSIRT(SIRT):
        def update_objective(self):
            residual = self.operator.direct(self.solution).as_array().astype(np.float64)
            residual -= self.data.as_array()
            self.loss.append(float(0.5 * np.sum(residual**2)))
            if not hasattr(self, 'loss_iterations'):
                self.loss_iterations = []
            self.loss_iterations.append(max(0, self.iteration))

    start = time.perf_counter()
    operator = ProjectionOperator(ig, data.geometry)
    solver = ResidualSIRT(initial=ig.allocate(0), operator=operator, data=data,
                          lower=0, update_objective_interval=10)
    def progress(algorithm):
        if algorithm.iteration % 100 == 0:
            print(f'SIRT iteration {algorithm.iteration}/{iterations}', flush=True)
    solver.run(iterations, callbacks=[progress], verbose=0)
    if solver.loss_iterations[-1] != solver.iteration:
        solver.update_objective()
    history = dict(iterations=solver.loss_iterations, loss=solver.loss,
                   projection_pixels=int(data.as_array().size),
                   definition='0.5 * sum((A x - b)^2), recomputed at current solution')
    elapsed = time.perf_counter() - start
    sirt_volume = np.asarray(solver.solution.as_array(), dtype=np.float32)
    return sirt_volume, history

if RUN_SIRT:
    reconstructed_volumes['sirt'], history = reconstruct_sirt(data, ig)


In [ ]:
if RUN_SIRT:
    steps = np.asarray(history['iterations'])
    mse = 2 * np.asarray(history['loss']) / history['projection_pixels']
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.semilogy(steps, mse)
    ax.set(xlabel='Completed iterations', ylabel='Projection residual MSE')
    ax.grid(alpha=0.3)
    plt.show()

# Axes show completed iterations and projection residual mean squared error.


### Inspect the current SIRT reconstruction

Display central slices along the three axes directly from the in-memory volume. The display window for an individual method may differ from the final shared window; use the shared-window comparison below to compare methods.


In [ ]:
if RUN_SIRT:
    methods = ['sirt']
    volumes = {m: reconstructed_volumes[m] for m in methods if m in reconstructed_volumes}
    if volumes:
        low, high = np.percentile(np.concatenate([v.ravel()[::16] for v in volumes.values()]), [1, 99.5])
        fig, axes = plt.subplots(len(volumes), 3, figsize=(12, 3.5 * len(volumes)), squeeze=False)
        for row, (method, volume) in enumerate(volumes.items()):
            for axis, label in enumerate(['Z', 'Y', 'X']):
                view = np.take(volume, volume.shape[axis] // 2, axis=axis)
                axes[row, axis].imshow(view, cmap='gray', origin='lower', vmin=low, vmax=high)
                axes[row, axis].set_title(f'{method.upper()} | {label} centre')
                axes[row, axis].axis('off')
        plt.tight_layout()
        plt.show()


## 7. Connect CIL data to FaCT-GS

As in the reference example connecting a PyTorch denoiser to CIL, **keep the core algorithm in its library and show the data adaptation and calls in the notebook**. We do not call the USB `convert.py`, `train.py` or dataset-reader scripts, or write projections to NPY files and reload them.

This section uses the in-memory projections, fixed splits and normalised geometry from Sections 2–4. Keep the trainer, camera model, configuration files and CUDA extensions in `third_party/fact-gs`: these are algorithm dependencies. Set `RUN_FACTGS = True` and run this section in order.

### Method background: how does FaCT-GS reconstruct CT?

FaCT-GS (Fast and Scalable CT Reconstruction with Gaussian Splatting), proposed by **Pawel Tomasz Pieta, Jakob Sauer Jørgensen and co-authors**, was published at **ECCV 2026**. It represents the attenuation field with optimisable 3D Gaussians and fits their parameters to measured X-ray projections. The paper focuses on the efficiency and scalability of projection rendering and voxelisation. [Paper](https://arxiv.org/abs/2604.01844) · [Project](https://papieta.github.io/fact-gs/) · [Official code](https://github.com/PaPieta/fact-gs)

**Representation.** The object is a sum of 3D Gaussians with different positions, sizes, orientations and densities:

$$
\mu(\mathbf{x}) = \sum_i \rho_i\exp\!\left[-\frac12(\mathbf{x}-\mathbf{p}_i)^\top\Sigma_i^{-1}(\mathbf{x}-\mathbf{p}_i)\right].
$$

Here $\mathbf{p}_i$ is position, $\rho_i$ is density amplitude and $\Sigma_i$ describes shape and orientation. These Gaussian parameters are optimised; this example requires neither a pretrained CNN nor 3D ground-truth supervision. [Method](https://arxiv.org/html/2604.01844v2#S3)

**From projections to a volume.** First reconstruct an FDK volume from the training projections, then sample initial Gaussians using its gradients. Repeatedly render a projection, compare it with the measured projection and backpropagate to update the Gaussians, with densification and pruning as configured. Finally, sample the Gaussian field onto a voxel grid to display 3D slices. The paper also supports initialisation from an existing volume prior; this example uses only FDK-gradient initialisation. [Initialisation and workflow](https://papieta.github.io/fact-gs/)

This example uses the bundled implementation's objective:

$$
\mathcal{L}=\mathcal{L}_{1}(\hat P,P)+0.3\,[1-\operatorname{SSIM}(\hat P,P)]+0.05\,\operatorname{TV}(\mu).
$$

The first two terms enforce projection consistency. TV penalises spatial variation and is evaluated on randomly sampled $32^3$ subvolumes in this implementation. Training uses only the training views; test views are reserved for evaluation. These weights and the initial Gaussian count are experiment settings, not established optimal parameters for USB data. See the [upstream trainer](third_party/fact-gs/train_recon.py).

**CIL's role.** CIL handles scan reading, preprocessing, geometry and the FDK/SIRT baselines. The adapter below passes the same projections and geometry to FaCT-GS. Unlike the reference notebook's denoiser inside FISTA, FaCT-GS is compared as an independent reconstruction method; it does not replace a SIRT step.

> Reference: Pawel Tomasz Pieta, Rasmus Juul Pedersen, Sina Borgi, Jakob Sauer Jørgensen, Jens Wenzel Andreasen, Vedrana Andersen Dahl. *FaCT-GS: Fast and Scalable CT Reconstruction with Gaussian Splatting*. ECCV 2026. [arXiv:2604.01844](https://arxiv.org/abs/2604.01844).

### 7.1 Import the implementation and set training parameters

Training runs in the current notebook kernel on the GPU selected in Section 1. Defaults are 20000 steps and 10000 initial Gaussians. Reduce `FACTGS_STEPS` for a short functionality check, but do not treat short training as a full experiment.

The upstream trainer still saves the final Gaussian model and volume, so an output directory is required. Visualisation and evaluation below use in-memory results rather than reloading these files.


In [ ]:
FACTGS_STEPS = 20000
FACTGS_GAUSSIANS = 10000
FACTGS_OUTPUT = CASE / 'runs_notebook' / f'{VIEWS}views_factgs'

def load_factgs():
    import sys
    repo = CASE / 'third_party/fact-gs'
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    os.environ['GLM_HOME'] = str(repo / 'fact_gs/submodules/glm')
    import tigre  # Import TIGRE before PyTorch.
    import torch
    return repo

if RUN_FACTGS:
    load_factgs()


### 7.2 Convert CIL geometry to FaCT-GS scanner parameters

Use CIL's TIGRE geometry conversion to obtain distances, detector parameters and corresponding rotation angles. Section 4 already normalised the geometry, so no additional scaling is applied here. Convert TIGRE voxel dimensions to the XYZ order expected by FaCT-GS.

This adapter supports the current circular cone-beam trajectory without tilt or offsets. Keep the checks below to avoid silently applying incorrect geometry to other scans.


In [ ]:
def make_factgs_scanner(data, ig, angles):
    from cil.plugins.tigre.Geometry import CIL2TIGREGeometry
    all_geometry = data.geometry.copy()
    all_geometry.set_angles(angles,
                            initial_angle=data.geometry.config.angles.initial_angle,
                            angle_unit=data.geometry.config.angles.angle_unit)
    geo, gaussian_angles = CIL2TIGREGeometry.getTIGREGeometry(ig, all_geometry)
    for offset in (geo.rotDetector, geo.offDetector, geo.offOrigin):
        if not np.allclose(offset, 0, atol=1e-6):
            raise ValueError('This FaCT-GS camera adapter supports only circular trajectories without tilt or offsets.')
    if not np.isclose(max(geo.sVoxel), 2):
        raise ValueError('Run Section 4.3 first to normalise the longest dimension to 2.')
    scanner = dict(
        mode='cone', DSO=float(geo.DSO), DSD=float(geo.DSD),
        nDetector=geo.nDetector.tolist(), sDetector=geo.sDetector.tolist(),
        dDetector=geo.dDetector.tolist(), nVoxel=geo.nVoxel[::-1].tolist(),
        sVoxel=geo.sVoxel[::-1].tolist(), dVoxel=geo.dVoxel[::-1].tolist(),
        offOrigin=[0, 0, 0], offDetector=[0, 0], accuracy=0.5, filter=None,
    )

    return scanner, gaussian_angles

if RUN_FACTGS:
    scanner, gaussian_angles = make_factgs_scanner(data, ig, angles)


### 7.3 Create camera interfaces from in-memory projections

Each training/test view contains an angle, camera pose and projection. `angle2pose` reuses the FaCT-GS pose convention. Only flip the input projections vertically to match its camera orientation; do not repeat the logarithm, downsampling or amplitude scaling. Flip the outputs back to CIL orientation afterwards.

`cil_scene_reader` supplies the trainer with training cameras, test cameras, scanner parameters and `None` for 3D ground truth. Its path argument is retained for interface compatibility, but it reads no files.


In [ ]:
def make_factgs_reader(scanner, gaussian_angles, projections, train_indices, test_indices):
    from fact_gs.r2_gaussian.dataset.dataset_readers import CameraInfo, SceneInfo, angle2pose
    camera_infos = {'train': [], 'test': []}
    fov_x = 2 * np.arctan2(scanner['sDetector'][1] / 2, scanner['DSD'])
    fov_y = 2 * np.arctan2(scanner['sDetector'][0] / 2, scanner['DSD'])
    for subset, indices in [('train', train_indices), ('test', test_indices)]:
        uid_offset = 0 if subset == 'train' else len(train_indices)
        for order, index in enumerate(indices):
            angle = float(gaussian_angles[index])
            world_to_camera = np.linalg.inv(angle2pose(scanner['DSO'], angle))
            camera_infos[subset].append(CameraInfo(
                uid=uid_offset + order,
                R=world_to_camera[:3, :3].T, T=world_to_camera[:3, 3],
                angle=angle, FovY=fov_y, FovX=fov_x,
                image=np.ascontiguousarray(projections[index, ::-1], dtype=np.float32),
                image_path=None, image_name=f'{int(index):04d}',
                width=scanner['nDetector'][1], height=scanner['nDetector'][0],
                mode=1, scanner_cfg=scanner,
            ))

    def cil_scene_reader(_path, evaluate):
        return SceneInfo(camera_infos['train'], camera_infos['test'] if evaluate else [],
                         None, scanner, 1.0)

    return cil_scene_reader

if RUN_FACTGS:
    cil_scene_reader = make_factgs_reader(scanner, gaussian_angles, projections, train_indices, test_indices)


### 7.4 Configure and run optimisation

Reuse the upstream configuration and explicitly set the experiment parameters: FDK-gradient initialisation, random seed 0 and `L1 + 0.3 × (1-SSIM) + 0.05 × TV`. FaCT-GS handles optimisation, backpropagation and densification; their implementation is not duplicated here.

The callback records losses and retains the final scene for rendering. Use a new output path to avoid overwriting earlier models. The upstream loop records indices from 0 through `FACTGS_STEPS`, with the last index corresponding to the final result.


In [ ]:
def train_factgs(cil_scene_reader, output, steps=20000, gaussians=10000):
    import random, torch
    from hydra import compose, initialize_config_dir
    from train_recon import optimize
    repo = load_factgs()
    with initialize_config_dir(config_dir=str(repo / 'config'), version_base=None):
        factgs_config = compose(config_name='default_recon', overrides=['eval=eval_silent'])
    factgs_config.model.data_source_path = 'in_memory_usb'
    factgs_config.model.model_path = str(output)
    factgs_config.model.num_gaussians = gaussians
    factgs_config.model.init_mode = 'gradient'
    factgs_config.model.density_thresh = 0.01
    factgs_config.model.eval = True
    factgs_config.optim.steps = steps
    factgs_config.optim.lambda_dssim = 0.3
    factgs_config.optim.lambda_tv = 0.05
    factgs_history = []
    factgs_result = {}

    def collect_factgs_result(step, losses, training_seconds, evaluation_seconds,
                             max_steps, eval_args, scene, render, voxelize,
                             init_mode, force_eval=False):
        if step % 100 == 0 or step == max_steps:
            factgs_history.append((step, float(losses['loss_total'])))
        if step == max_steps:
            factgs_result['scene'] = scene
            factgs_result['training_seconds'] = training_seconds

    output.mkdir(parents=True, exist_ok=False)
    random.seed(0)
    np.random.seed(0)
    torch.manual_seed(0)
    torch.cuda.set_device(0)  # The first GPU exposed by CUDA_VISIBLE_DEVICES.
    start = time.perf_counter()
    optimize(factgs_config, scene_reader=cil_scene_reader,
             report_callback=collect_factgs_result)
    print(f'FaCT-GS elapsed time: {time.perf_counter() - start:.3f} seconds')

    return factgs_result['scene'], factgs_history

if RUN_FACTGS:
    scene, factgs_history = train_factgs(cil_scene_reader, FACTGS_OUTPUT, FACTGS_STEPS, FACTGS_GAUSSIANS)


### 7.5 Obtain and display in-memory volumes and projections

Convert the Gaussian volume from XYZ to CIL's ZYX order. Render training/test projections with the Gaussian renderer, order them by their original indices and flip them back to CIL's vertical orientation. Store the results in `reconstructed_volumes` and `factgs_projections` for the shared evaluation and plots in Sections 8 and 9.


In [ ]:
def render_factgs_results(scene, scanner, projections, train_indices, test_indices):
    import torch
    from fact_gs import rasterize_proj, voxelize_vol
    with torch.no_grad():
        volume_xyz = voxelize_vol(scene.gaussians, scanner['offOrigin'],
                                 scanner['nVoxel'], scanner['sVoxel'])['vol']
        volume = np.ascontiguousarray(
            volume_xyz.detach().cpu().numpy().transpose(2, 1, 0), dtype=np.float32)
        factgs_projections = {}
        for subset, indices, cameras in [
            ('train', train_indices, scene.getTrainCameras()),
            ('test', test_indices, scene.getTestCameras()),
        ]:
            by_index = {int(camera.image_name): camera for camera in cameras}
            predicted = []
            for index in indices:
                image = rasterize_proj(by_index[int(index)], scene.gaussians)['render']
                predicted.append(image.squeeze().detach().cpu().numpy()[::-1].copy())
            factgs_projections[subset] = (projections[indices], np.stack(predicted))
    return volume, factgs_projections

if RUN_FACTGS:
    reconstructed_volumes['factgs'], factgs_projections = render_factgs_results(
        scene, scanner, projections, train_indices, test_indices)
    steps, losses = np.asarray(factgs_history).T
    plt.plot(steps, losses)
    plt.xlabel('Training step')
    plt.ylabel('FaCT-GS total loss')
    plt.show()

def reconstruct_factgs(data, ig, angles, projections, train_indices, test_indices,
                       output, steps=20000, gaussians=10000):
    load_factgs()
    scanner, gaussian_angles = make_factgs_scanner(data, ig, angles)
    reader = make_factgs_reader(scanner, gaussian_angles, projections, train_indices, test_indices)
    scene, history = train_factgs(reader, output, steps, gaussians)
    return render_factgs_results(scene, scanner, projections, train_indices, test_indices)


In [ ]:
if RUN_FACTGS:
    methods = ['factgs']
    volumes = {m: reconstructed_volumes[m] for m in methods if m in reconstructed_volumes}
    if volumes:
        low, high = np.percentile(np.concatenate([v.ravel()[::16] for v in volumes.values()]), [1, 99.5])
        fig, axes = plt.subplots(len(volumes), 3, figsize=(12, 3.5 * len(volumes)), squeeze=False)
        for row, (method, volume) in enumerate(volumes.items()):
            for axis, label in enumerate(['Z', 'Y', 'X']):
                view = np.take(volume, volume.shape[axis] // 2, axis=axis)
                axes[row, axis].imshow(view, cmap='gray', origin='lower', vmin=low, vmax=high)
                axes[row, axis].set_title(f'{method.upper()} | {label} centre')
                axes[row, axis].axis('off')
        plt.tight_layout()
        plt.show()


## 8. Evaluate reconstruction using training and test projections

Now create the test geometry: keep the source, detector and coordinate scale unchanged and replace only the angles. Forward-project the reconstructed in-memory volumes and compare with the measured projections at the same angles. Test projections were excluded from reconstruction.

Proceed in order: select methods, generate projections and compute metrics. Select only methods completed in this run; do not load NPY or PNG files. FaCT-GS uses the Gaussian-rendered projections from Section 7; FDK/SIRT use CIL/TIGRE forward projection.


In [ ]:
def project_cil_volume(volume, data, ig, projections, angles, test_indices):
    from cil.plugins.tigre import ProjectionOperator
    test_geometry = data.geometry.copy()
    test_geometry.set_angles(angles[test_indices],
                            initial_angle=data.geometry.config.angles.initial_angle,
                            angle_unit=data.geometry.config.angles.angle_unit)
    image = ig.allocate(0)
    image.fill(volume)
    result = {}
    for subset, measured, geometry in [
        ('train', data.as_array(), data.geometry),
        ('test', projections[test_indices], test_geometry),
    ]:
        predicted = ProjectionOperator(ig, geometry).direct(image).as_array()
        result[subset] = (measured, predicted)
    return result

METHODS_TO_EVALUATE = list(reconstructed_volumes)
if RUN_METRICS:
    projection_results = {}
    for method in METHODS_TO_EVALUATE:
        projection_results[method] = (factgs_projections if method == 'factgs' else
            project_cil_volume(reconstructed_volumes[method], data, ig, projections, angles, test_indices))


### Compute PSNR and SSIM per view

Compute the metrics for each view and average them. Use the historical experiments' shared `data_range = 3.142874300`, a uniform 7×7 SSIM window and sample covariance. This measures projection consistency, not error against 3D ground truth.


In [ ]:
def evaluate_projections(pairs):
    from skimage.metrics import peak_signal_noise_ratio, structural_similarity
    scores = {}
    for subset, (measured, predicted) in pairs.items():
        psnr = [peak_signal_noise_ratio(a, b, data_range=3.142874300)
                for a, b in zip(measured, predicted)]
        ssim = [structural_similarity(a, b, data_range=3.142874300,
                                     win_size=7, gaussian_weights=False,
                                     use_sample_covariance=True)
                for a, b in zip(measured, predicted)]
        scores[subset] = dict(psnr_mean_db=float(np.mean(psnr)), ssim_mean=float(np.mean(ssim)))
    return scores

if RUN_METRICS:
    metric_results = {method: evaluate_projections(pairs)
                      for method, pairs in projection_results.items()}


In [ ]:
if RUN_METRICS:
    rows = ['| Method | Train PSNR | Train SSIM | Test PSNR | Test SSIM |',
            '| --- | ---: | ---: | ---: | ---: |']
    for method in METHODS_TO_EVALUATE:
        q = metric_results[method]
        a, b = q['train'], q['test']
        rows.append(f"| {method} | {a['psnr_mean_db']:.4f} | {a['ssim_mean']:.4f} | {b['psnr_mean_db']:.4f} | {b['ssim_mean']:.4f} |")
    display(Markdown('\n'.join(rows)))

## 9. Compare and interpret the results

### In-memory reconstructions: slices with a shared display window

Display the volumes currently in `reconstructed_volumes`, without reading from disk. All slices share a display window; the numerical volumes are neither clipped nor normalised. Rerunning Section 1 clears these results. After changing views or geometry, rerun reconstruction rather than reusing old volumes.


In [ ]:
methods = list(reconstructed_volumes)
volumes = {m: reconstructed_volumes[m] for m in methods if m in reconstructed_volumes}
if volumes:
    low, high = np.percentile(np.concatenate([v.ravel()[::16] for v in volumes.values()]), [1, 99.5])
    fig, axes = plt.subplots(len(volumes), 3, figsize=(12, 3.5 * len(volumes)), squeeze=False)
    for row, (method, volume) in enumerate(volumes.items()):
        for axis, label in enumerate(['Z', 'Y', 'X']):
            view = np.take(volume, volume.shape[axis] // 2, axis=axis)
            axes[row, axis].imshow(view, cmap='gray', origin='lower', vmin=low, vmax=high)
            axes[row, axis].set_title(f'{method.upper()} | {label} centre')
            axes[row, axis].axis('off')
    plt.tight_layout()
    plt.show()


### Historical experiments: 25 / 50 / 75 views

The following are fixed figures supplied with the project, **not results generated by the current execution above**. Rows correspond to 75, 50 and 25 training views, from top to bottom.

![Historical central slices](assets/factgs_middle_slice.png)

Columns show FDK, SIRT and FaCT-GS, from left to right. Inspect structural boundaries, streaks and local detail as the number of views decreases. All slices use a shared display window.

![Historical test-projection comparison](assets/test_projection_0252.png)

The test view has original index 252 and was excluded from training. Columns show the measured projection, FDK, SIRT and FaCT-GS. This compares prediction at an unseen angle, not 3D ground truth.

In the historical 25-view experiment, test-projection PSNR/SSIM were **22.2558 / 0.4308** for FDK, **26.9690 / 0.6383** for SIRT and **27.3968 / 0.9361** for FaCT-GS. FaCT-GS achieved higher test-projection metrics in that setting; this does not establish that its 3D structure is the most accurate. See the [experiment record](EXPERIMENT_RECORD.md) for the full results, timing definitions and parameters.

### Limitations and further experiments

- No 3D ground truth is available. PSNR/SSIM measure consistency with noisy projections, not 3D reconstruction error.
- FDK/SIRT use the CIL/TIGRE volume projector, while FaCT-GS uses a Gaussian renderer; their projection computations differ.
- SIRT uses 1000 iterations and FaCT-GS uses 20000 steps. These are not equal computational budgets and do not guarantee convergence.
- The historical metrics use a shared `data_range`. Determine it again for a different dataset rather than copying this value.
- Further experiments could compare random seeds and iteration budgets, and use simulated data with known 3D ground truth to validate geometric accuracy.


## 10. Run all three view counts

First run Sections 1–8 in order to define the functions; all individual reconstruction switches may remain disabled. Then set `RUN_ALL_VIEWS = True` below to run 25, 50 and 75 views, each independently initialised and sharing the same test set.

`batch_results[view_count]` stores volumes and metrics for each method. All runs reuse the same preprocessed data; FaCT-GS outputs are stored in separate directories for each view count.


In [ ]:
RUN_ALL_VIEWS = False
VIEW_COUNTS = [25, 50, 75]
BATCH_METHODS = ['fdk', 'sirt', 'factgs']

if RUN_ALL_VIEWS:
    from datetime import datetime
    batch_output = CASE / 'runs_notebook' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    batch_results = {}
    for view_count in VIEW_COUNTS:
        indices = np.asarray(train_splits[view_count])
        group_data = select_training_data(preprocessed, indices)
        _, sizes, counts = make_volume_grid(group_data.geometry)
        group_data, group_ig = normalize_geometry(group_data, sizes, counts)
        volumes, metrics = {}, {}
        batch_results[view_count] = dict(volumes=volumes, metrics=metrics)
        for method in BATCH_METHODS:
            print(f'{view_count} views | {method.upper()}')
            if method == 'fdk':
                volume = reconstruct_fdk(group_data, group_ig)
            elif method == 'sirt':
                volume, _ = reconstruct_sirt(group_data, group_ig, iterations=1000)
            elif method == 'factgs':
                volume, pairs = reconstruct_factgs(
                    group_data, group_ig, angles, projections, indices, test_indices,
                    batch_output / f'{view_count}views' / 'factgs',
                    steps=FACTGS_STEPS, gaussians=FACTGS_GAUSSIANS)
            else:
                raise ValueError(f'Unknown method: {method}')
            if method != 'factgs':
                pairs = project_cil_volume(volume, group_data, group_ig, projections, angles, test_indices)
            volumes[method] = volume
            metrics[method] = evaluate_projections(pairs)


### Summarise the current batch metrics

List only methods completed in this batch.


In [ ]:
if RUN_ALL_VIEWS:
    rows = ['| Training views | Method | Train PSNR | Train SSIM | Test PSNR | Test SSIM |',
            '| ---: | --- | ---: | ---: | ---: | ---: |']
    for view_count, result in batch_results.items():
        for method, scores in result['metrics'].items():
            train, test = scores['train'], scores['test']
            rows.append(
                f"| {view_count} | {method.upper()} | {train['psnr_mean_db']:.4f} | "
                f"{train['ssim_mean']:.4f} | {test['psnr_mean_db']:.4f} | {test['ssim_mean']:.4f} |")
    display(Markdown('\n'.join(rows)))


### Compare slices across the three view counts

Rows correspond to view counts and columns to reconstruction methods, with a shared display window.


In [ ]:
if RUN_ALL_VIEWS:
    samples = np.concatenate([
        volume.ravel()[::16]
        for result in batch_results.values() for volume in result['volumes'].values()
    ])
    low, high = np.percentile(samples, [1, 99.5])
    fig, axes = plt.subplots(len(VIEW_COUNTS), len(BATCH_METHODS),
                             figsize=(4 * len(BATCH_METHODS), 3 * len(VIEW_COUNTS)),
                             squeeze=False)
    for row, view_count in enumerate(VIEW_COUNTS):
        for col, method in enumerate(BATCH_METHODS):
            volume = batch_results[view_count]['volumes'][method]
            axes[row, col].imshow(volume[volume.shape[0] // 2],
                                  cmap='gray', origin='lower', vmin=low, vmax=high)
            axes[row, col].set_title(f'{view_count} views | {method.upper()}')
            axes[row, col].axis('off')
    plt.tight_layout()
    plt.show()
